# Alternative Protein Consumer Psychology

## Computational Discovery of Psychological Barriers in Alternative‑Protein Discourse

**Author:** Anuj Pal  
**Date:** August 2026  
**Purpose:** Proof‑of‑concept for Study 1 of the doctoral research proposal *"Psychological Barriers to Sustainable Protein Adoption: A Computational‑to‑Experimental Approach"*

---

### Research Context

This notebook implements a computational pipeline to identify psychological signals in naturally occurring consumer discourse about alternative‑protein products. The work is informed by research on:

- **The Differentiation Principle** (Florack et al., 2021): Consumers may focus disproportionately on attributes that distinguish a novel product from a familiar alternative.
- **Sustainability Liability** (Kunz et al., 2021): Sustainability attributes can produce positive or negative differentiation depending on context.
- **Perceived Naturalness** (Steiner et al., 2026): Perceived processing shapes naturalness evaluations of plant‑based meat alternatives.

The computational results are treated as **hypothesis‑generating** rather than causal evidence. Their primary purpose is to identify candidate psychological mechanisms for subsequent experimental validation.

---

### Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Imports, configuration, helper functions |
| 2 | Data loading and platform mapping |
| 3 | Zero‑shot classifier initialization |
| 4 | Batch classification execution |
| 5 | Aggregate report and dominant construct analysis |
| 6 | Cross‑platform comparison (Mann‑Whitney U test) |
| 7 | KeyBERT evidence extraction (sample texts) |
| 8 | BERTopic theme extraction |
| 9 | Top discriminative keywords per construct |
| 10 | Mechanism → Intervention mapping |
| 11 | Psychologically tailored intervention recommendations |
| 12 | Save results |

## Research Objectives

The pipeline is designed to:

1. **Identify** recurring psychological signals in alternative‑protein consumer discourse.
2. **Operationalise** constructs relevant to novel‑product evaluation and adoption.
3. **Examine** how these signals vary across consumer discussions and platforms (X/Twitter vs. Reddit).
4. **Generate** interpretable textual evidence associated with identified constructs (KeyBERT).
5. **Discover** broader thematic patterns using topic modelling (BERTopic).
6. **Translate** exploratory findings into candidate hypotheses and communication interventions for subsequent experimental testing.

## Psychological Construct Taxonomy

The current exploratory taxonomy contains **ten constructs**, each operationalised as a natural‑language hypothesis for zero‑shot classification.

| Construct | Operational Focus |
|-----------|-------------------|
| `differentiation_from_meat` | Comparisons between alternative proteins and conventional meat |
| `perceived_processing` | Concerns about ultra‑processing, additives, or industrial production |
| `naturalness_perceptions` | Perceptions of products as artificial, synthetic, fake, or unnatural |
| `taste_expectations` | Expectations regarding taste, texture, and sensory quality |
| `sustainability_differentiation` | Evaluation or questioning of environmental and sustainability claims |
| `anchoring_to_meat` | Use of conventional meat as the default benchmark |
| `perceived_loss_switching` | Expected losses in enjoyment, satiety, nutrition, or other valued attributes |
| `trust_and_credibility` | Concerns about transparency, corporate motives, and credibility |
| `uncertainty_and_risk` | Uncertainty concerning health, safety, ingredients, or longer‑term risks |
| `social_acceptance` | Concerns about social judgement, family acceptance, and cultural norms |

**Important:** These constructs are **operational hypotheses**, not validated psychological scales. The scores produced by the classifier are model‑assigned probabilities, not prevalence estimates.

## 1. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import mannwhitneyu
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

try:
    from keybert import KeyBERT
    KEYBERT_AVAILABLE = True
except ImportError:
    KEYBERT_AVAILABLE = False
    print("KeyBERT not installed. Install with: pip install keybert")

# File paths
CSV_PATH = r"data\alt_protein_discussions.csv"
OUTPUT_PATH = r"data\alt_protein_results.csv"
SUMMARY_PATH = r"data\alt_protein_summary.csv"

# constructs and their natural-language hypotheses
DISPLAY_LABELS = [
    "differentiation_from_meat",
    "perceived_processing",
    "naturalness_perceptions",
    "taste_expectations",
    "sustainability_differentiation",
    "anchoring_to_meat",
    "perceived_loss_switching",
    "trust_and_credibility",
    "uncertainty_and_risk",
    "social_acceptance",
]

HYPOTHESIS_LABELS = [
    "The consumer focuses on how this alternative product differs from or compares unfavorably to conventional meat.",
    "The consumer expresses concern over ultra-processing, heavy additives, or industrial chemical manufacturing.",
    "The consumer perceives this alternative protein as artificial, synthetic, fake, or unnatural.",
    "The consumer expects unappealing taste, rubbery texture, or poor sensory quality compared to meat.",
    "The consumer evaluates or questions the environmental, climate, and sustainability claims of the product.",
    "The consumer uses real conventional meat as the default benchmark and resists changing established eating habits.",
    "The consumer fears losing enjoyment, satiety, or nutritional value when replacing conventional meat.",
    "The consumer doubts producer transparency, corporate motivations, or clean labeling.",
    "The consumer expresses uncertainty regarding long-term health safety, digestibility, or unknown ingredients.",
    "The consumer worries about social judgment, family rejection, or violating cultural dining traditions.",
]

LABEL_MAP = dict(zip(DISPLAY_LABELS, HYPOTHESIS_LABELS))

def safe_bar(value):
    """Create a bar string for visual display of percentages."""
    if pd.isna(value) or np.isnan(value):
        return " " * 20
    val = int(value)
    return "█" * val + "░" * (20 - val) if val <= 20 else "█" * 20

## 2. Data Loading and Platform Mapping

In [ ]:

print("=" * 70)
print("   ALTERNATIVE PROTEIN CONSUMER PSYCHOLOGY PIPELINE")
print("   Proof-of-Concept for Study 1: Computational Analysis of Consumer Discourse")
print("=" * 70)

print("\n[1] Loading data...")
df = pd.read_csv(CSV_PATH)
print(f"    Loaded {len(df)} alternative-protein consumer discussions.")

# Map source to platform
df['platform'] = df['source'].apply(lambda x: 'Twitter' if x == 'X' else 'Reddit')

texts = df["text"].dropna().astype(str).tolist()
print(f"    {len(texts)} valid texts ready for computational analysis.")

   ALTERNATIVE PROTEIN CONSUMER PSYCHOLOGY PIPELINE
   Proof-of-Concept for Study 1: Computational Analysis of Consumer Discourse

[1] Loading data...
    Loaded 123 alternative-protein consumer discussions.
    123 valid texts ready for computational analysis.


## 3. Zero-Shot Classifier Initialization

In [ ]:
#  (BART-Large-MNLI)
print("\n[2] Loading zero-shot NLI classifier (BART-Large-MNLI)...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    return_all_scores=True,
    device=0,  # Use GPU; change to -1 if CPU
    batch_size=8,
)
print("    Classifier loaded successfully.")
print("\n    Target Constructs:")
for i, label in enumerate(DISPLAY_LABELS):
    print(f"      {i+1}. {label:<30}: {HYPOTHESIS_LABELS[i]}")

def predict_construct_probabilities(texts_batch):
    """Run zero-shot classification on a batch of texts, returning probabilities for each construct."""
    if isinstance(texts_batch, str):
        texts_batch = [texts_batch]
    results = classifier(texts_batch, HYPOTHESIS_LABELS, multi_label=True)
    if isinstance(results, dict):
        results = [results]
    probs = []
    for res in results:
        label_scores = {label: score for label, score in zip(res["labels"], res["scores"])}
        probs.append([label_scores[hyp] for hyp in HYPOTHESIS_LABELS])
    return np.array(probs)


[2] Loading zero-shot NLI classifier (BART-Large-MNLI)...


Device set to use cuda:0


    Classifier loaded successfully.

    Target Constructs:
      1. differentiation_from_meat     : The consumer focuses on how this alternative product differs from or compares unfavorably to conventional meat.
      2. perceived_processing          : The consumer expresses concern over ultra-processing, heavy additives, or industrial chemical manufacturing.
      3. naturalness_perceptions       : The consumer perceives this alternative protein as artificial, synthetic, fake, or unnatural.
      4. taste_expectations            : The consumer expects unappealing taste, rubbery texture, or poor sensory quality compared to meat.
      5. sustainability_differentiation: The consumer evaluates or questions the environmental, climate, and sustainability claims of the product.
      6. anchoring_to_meat             : The consumer uses real conventional meat as the default benchmark and resists changing established eating habits.
      7. perceived_loss_switching      : The consumer fears 

## 4. Batch Classification Execution

In [ ]:

print("\n[3] Executing zero-shot classification on alternative protein corpus...")
batch_size = 16
all_probs = []
total_batches = (len(texts) + batch_size - 1) // batch_size

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    probs = predict_construct_probabilities(batch)
    all_probs.append(probs)
    print(f"    Processed batch {i//batch_size + 1}/{total_batches}")

probs_array = np.vstack(all_probs)
print(f"    Classification complete for {len(texts)} texts.")

# Add probabilities to dataframe
for i, col in enumerate(DISPLAY_LABELS):
    df[col] = probs_array[:, i]


[3] Executing zero-shot classification on alternative protein corpus...
    Processed batch 1/8
    Processed batch 2/8
    Processed batch 3/8
    Processed batch 4/8
    Processed batch 5/8
    Processed batch 6/8
    Processed batch 7/8
    Processed batch 8/8
    Classification complete for 123 texts.


## 5. Aggregate Report and Dominant Construct Analysis

In [5]:
mean_probs = probs_array.mean(axis=0)
std_probs = probs_array.std(axis=0)

print("\n" + "=" * 70)
print(" AGGREGATE PSYCHOLOGICAL BARRIER REPORT (ALTERNATIVE PROTEINS)")
print("=" * 70)
print(f"\nOverall Construct Prevalence across texts (n = {len(texts)}):")
print("-" * 65)

for i, label in enumerate(DISPLAY_LABELS):
    mean_pct = mean_probs[i] * 100
    std_pct = std_probs[i] * 100
    bar = safe_bar(mean_pct / 5)
    print(f"  {label:<32}: {mean_pct:5.1f}%  (± {std_pct:.1f}%)  {bar}")

# Significant barriers (>40%)
print("\n[4] Significant Psychological Barriers (>40% prevalence):")
for label, score in zip(DISPLAY_LABELS, mean_probs):
    if score > 0.4:
        print(f"  ✅ {label}: {score*100:.1f}%")

# Dominant construct per text
dominant_indices = np.argmax(probs_array, axis=1)
df["dominant_construct"] = [DISPLAY_LABELS[idx] for idx in dominant_indices]
dominant_counts = df["dominant_construct"].value_counts()

print("\n" + "-" * 65)
print("Dominant Construct Distribution across Corpus:")
for construct, count in dominant_counts.items():
    percentage = (count / len(texts)) * 100
    print(f"  {construct:<32}: {count:>3} texts ({percentage:.1f}%)")


 AGGREGATE PSYCHOLOGICAL BARRIER REPORT (ALTERNATIVE PROTEINS)

Overall Construct Prevalence across texts (n = 123):
-----------------------------------------------------------------
  differentiation_from_meat       :  41.4%  (± 31.7%)  ████████░░░░░░░░░░░░
  perceived_processing            :  29.3%  (± 31.0%)  █████░░░░░░░░░░░░░░░
  naturalness_perceptions         :  30.7%  (± 35.5%)  ██████░░░░░░░░░░░░░░
  taste_expectations              :  11.8%  (± 19.2%)  ██░░░░░░░░░░░░░░░░░░
  sustainability_differentiation  :  27.5%  (± 31.6%)  █████░░░░░░░░░░░░░░░
  anchoring_to_meat               :   1.1%  (± 3.4%)  ░░░░░░░░░░░░░░░░░░░░
  perceived_loss_switching        :  23.8%  (± 25.2%)  ████░░░░░░░░░░░░░░░░
  trust_and_credibility           :  15.1%  (± 21.9%)  ███░░░░░░░░░░░░░░░░░
  uncertainty_and_risk            :  53.1%  (± 38.3%)  ██████████░░░░░░░░░░
  social_acceptance               :  11.2%  (± 14.4%)  ██░░░░░░░░░░░░░░░░░░

[4] Significant Psychological Barriers (>40% prevalence)

## 6. Cross-Platform Comparison (Mann-Whitney U)
This tests whether construct scores differ significantly between X/Twitter and Reddit.

In [ ]:
if 'platform' in df.columns:
    print("\n[5] Cross-Platform Comparison (Twitter vs Reddit):")
    twitter_data = df[df['platform'] == 'Twitter']
    reddit_data = df[df['platform'] == 'Reddit']
    
    for label in DISPLAY_LABELS:
        if len(twitter_data) > 0 and len(reddit_data) > 0:
            stat, p = mannwhitneyu(twitter_data[label], reddit_data[label])
            if p < 0.05:
                print(f"  {label}: Significant difference (p = {p:.4f})")
    print("  (Only significant differences are shown.)")


[5] Cross-Platform Comparison (Twitter vs Reddit):
  sustainability_differentiation: Significant difference (p = 0.0029)
  (Only significant differences are shown.)


## 7. KeyBERT Evidence Extraction (Sample Texts)
This provides interpretable keyword attribution supporting the construct classifications.

In [7]:
print("\n[6] Generating interpretable keyword attribution with KeyBERT...")
if KEYBERT_AVAILABLE:
    kw_model = KeyBERT()

sample_indices = np.random.choice(len(texts), min(3, len(texts)), replace=False)

print("\n" + "=" * 70)
print(" SAMPLE TEXT EVIDENCE & CONSTRUCT ATTRIBUTION")
print("=" * 70)

for idx in sample_indices:
    text = texts[idx]
    print(f"\nSAMPLE TEXT:\n'{text}'\n")
    
    scores = df.loc[idx, DISPLAY_LABELS].values.astype(float)
    dom_idx = np.argmax(scores)
    dom_label = DISPLAY_LABELS[dom_idx]
    
    print("Construct Probabilities:")
    for i, label in enumerate(DISPLAY_LABELS):
        pct = scores[i] * 100
        print(f"  {label:<30}: {pct:5.1f}% {safe_bar(pct/5)}")
    
    print(f"\n  DOMINANT BARRIER: {dom_label.upper()} ({scores[dom_idx]*100:.1f}%)")
    
    if KEYBERT_AVAILABLE:
        keywords = kw_model.extract_keywords(
            text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=4
        )
        print("  Extracted Supporting Keyphrases:")
        for kw, score in keywords:
            print(f"    • '{kw}' (semantic relevance: {score:.3f})")
    print("-" * 50)


[6] Generating interpretable keyword attribution with KeyBERT...

 SAMPLE TEXT EVIDENCE & CONSTRUCT ATTRIBUTION

SAMPLE TEXT:
'For the occasional biannual trip to BK, I prefer the impossible whopper over the regular one. For the occasional frozen nugget... Quorn nuggets...'

Construct Probabilities:
  differentiation_from_meat     :  29.3% █████░░░░░░░░░░░░░░░
  perceived_processing          :   9.2% █░░░░░░░░░░░░░░░░░░░
  naturalness_perceptions       :   3.1% ░░░░░░░░░░░░░░░░░░░░
  taste_expectations            :   1.2% ░░░░░░░░░░░░░░░░░░░░
  sustainability_differentiation:   1.4% ░░░░░░░░░░░░░░░░░░░░
  anchoring_to_meat             :   0.2% ░░░░░░░░░░░░░░░░░░░░
  perceived_loss_switching      :  17.4% ███░░░░░░░░░░░░░░░░░
  trust_and_credibility         :   1.8% ░░░░░░░░░░░░░░░░░░░░
  uncertainty_and_risk          :  61.0% ████████████░░░░░░░░
  social_acceptance             :   3.9% ░░░░░░░░░░░░░░░░░░░░

  DOMINANT BARRIER: UNCERTAINTY_AND_RISK (61.0%)
  Extracted Supporting Keyph

## 8. BERTopic Theme Extraction
This provides a complementary, less theory-constrained view of recurring discourse themes.

In [9]:
print("\n[7] Extracting thematic patterns with BERTopic...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model, nr_topics="auto", verbose=True)
topics, probs = topic_model.fit_transform(texts)
topic_info = topic_model.get_topic_info()
print("\nDiscovered Themes in Alternative-Protein Discourse:")
print(topic_info[['Topic', 'Count', 'Name']].head(10))


[7] Extracting thematic patterns with BERTopic...


2026-08-13 21:42:25,484 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2026-08-13 21:42:25,899 - BERTopic - Embedding - Completed ✓
2026-08-13 21:42:25,899 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-13 21:42:39,915 - BERTopic - Dimensionality - Completed ✓
2026-08-13 21:42:39,921 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-13 21:42:39,941 - BERTopic - Cluster - Completed ✓
2026-08-13 21:42:39,942 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-13 21:42:39,976 - BERTopic - Representation - Completed ✓
2026-08-13 21:42:39,977 - BERTopic - Topic reduction - Reducing number of topics
2026-08-13 21:42:39,990 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-13 21:42:40,015 - BERTopic - Representation - Completed ✓
2026-08-13 21:42:40,018 - BERTopic - Topic reduction - Reduced number of topics from 3 to 3



Discovered Themes in Alternative-Protein Discourse:
   Topic  Count                  Name
0     -1     14     -1_and_to_it_meat
1      0     61     0_the_and_it_like
2      1     48  1_meat_grown_lab_the


## 9. Top Discriminative Keywords per Construct
This shows which keywords are most associated with each psychological construct.

In [10]:
print("\n[8] Top Discriminative Keywords for Each Psychological Construct:")
print("=" * 70)

if KEYBERT_AVAILABLE and 'dominant_construct' in df.columns:
    for label in DISPLAY_LABELS:
        dominant_texts = df[df['dominant_construct'] == label]['text'].tolist()
        if len(dominant_texts) >= 3:
            sample_texts = dominant_texts[:5]
            keywords = []
            for text in sample_texts:
                try:
                    kw = kw_model.extract_keywords(
                        text, keyphrase_ngram_range=(1,2), stop_words='english', top_n=3
                    )
                    keywords.extend([k[0] for k in kw])
                except:
                    continue
            unique_kw = []
            for k in keywords:
                if k not in unique_kw:
                    unique_kw.append(k)
            unique_kw = unique_kw[:5]
            if unique_kw:
                print(f"\n  {label.upper()} (n={len(dominant_texts)} texts):")
                print(f"    → {', '.join(unique_kw)}")
        else:
            print(f"\n  {label.upper()}: insufficient texts for keyword extraction")


[8] Top Discriminative Keywords for Each Psychological Construct:

  DIFFERENTIATION_FROM_MEAT (n=27 texts):
    → meat environmentally, grown meat, animal meat, meat environmental, meat biologically

  PERCEIVED_PROCESSING: insufficient texts for keyword extraction

  NATURALNESS_PERCEPTIONS (n=8 texts):
    → taste nutrients, nutrients feeling, taste, burger great, burger

  TASTE_EXPECTATIONS: insufficient texts for keyword extraction

  SUSTAINABILITY_DIFFERENTIATION (n=18 texts):
    → grown meat, meat meat, meat, natural meat, lab grown

  ANCHORING_TO_MEAT: insufficient texts for keyword extraction

  PERCEIVED_LOSS_SWITCHING (n=4 texts):
    → veggie meats, love veggie, meat vegetarian, steak, steak love

  TRUST_AND_CREDIBILITY: insufficient texts for keyword extraction

  UNCERTAINTY_AND_RISK (n=64 texts):
    → grown meat, protein oils, meat pea, lab grown, meat reasons

  SOCIAL_ACCEPTANCE: insufficient texts for keyword extraction


##  10. Mechanism → Intervention Mapping (Dynamic)
This table maps identified psychological barriers to potential interventions for Study 3.

In [11]:
print("\n" + "=" * 70)
print(" PSYCHOLOGICAL MECHANISM → INTERVENTION MAPPING")
print("=" * 70)

intervention_map = {
    "uncertainty_and_risk": "Third-party safety certifications; transparent ingredient disclosure; independent lab results",
    "differentiation_from_meat": "Positive framing of unique attributes; avoid direct comparison where product falls short",
    "naturalness_perceptions": "Reduce packaging colour saturation (Steiner et al., 2026); emphasise plant-origin ingredients",
    "perceived_processing": "Highlight simple, transparent processing methods; reframe as natural fermentation or food craft",
    "sustainability_differentiation": "Pair sustainability claims with taste and quality assurances (Kunz et al., 2021)",
    "taste_expectations": "Sensory-rich descriptions; chef-led culinary preparation tips",
    "perceived_loss_switching": "Emphasise satisfaction, satiety, and nutritional adequacy",
    "trust_and_credibility": "Independent third-party certifications; transparent LCA data",
    "social_acceptance": "Highlight growing adoption; peer testimonials; cultural fit",
    "anchoring_to_meat": "Frame as complementary addition; flexitarian integration"
}

sorted_for_table = sorted(zip(DISPLAY_LABELS, mean_probs), key=lambda x: x[1], reverse=True)
top_constructs = sorted_for_table[:5]

print("\n  +-----------------------------+--------------------------------------------------+")
print("  | Psychological Mechanism     | Potential Intervention                           |")
print("  +-----------------------------+--------------------------------------------------+")

for label, score in top_constructs:
    display_name = label.replace('_', ' ').title()
    intervention = intervention_map.get(label, "Provide targeted informational framing addressing consumer risk")
    
    # Word wrap for table display
    words = intervention.split()
    lines = []
    current_line = ""
    for word in words:
        if len(current_line) + len(word) + 1 <= 48:
            current_line += " " + word if current_line else word
        else:
            lines.append(current_line)
            current_line = word
    if current_line:
        lines.append(current_line)
    
    print(f"  | {display_name:<27} | {lines[0]:<48} |")
    for line in lines[1:]:
        print(f"  | {'':27} | {line:<48} |")
    print("  +-----------------------------+--------------------------------------------------+")


 PSYCHOLOGICAL MECHANISM → INTERVENTION MAPPING

  +-----------------------------+--------------------------------------------------+
  | Psychological Mechanism     | Potential Intervention                           |
  +-----------------------------+--------------------------------------------------+
  | Uncertainty And Risk        | Third-party safety certifications; transparent   |
  |                             | ingredient disclosure; independent lab results   |
  +-----------------------------+--------------------------------------------------+
  | Differentiation From Meat   | Positive framing of unique attributes; avoid     |
  |                             | direct comparison where product falls short      |
  +-----------------------------+--------------------------------------------------+
  | Naturalness Perceptions     | Reduce packaging colour saturation (Steiner et   |
  |                             | al., 2026); emphasise plant-origin ingredients   |
  +------------

## 11. Psychologically Tailored Intervention Recommendations
This provides concrete communication strategies for addressing the top identified barriers.

In [12]:
print("\n" + "=" * 70)
print(" PSYCHOLOGICALLY TAILORED INTERVENTION RECOMMENDATIONS (STUDY 3)")
print("=" * 70)

interventions = {
    "differentiation_from_meat": (
        "• Apply positive differentiation framing (emphasize unique culinary"
        " strengths rather than mimicking meat perfectly).\n• Avoid inviting"
        " direct organoleptic comparison where the product falls short."
    ),
    "perceived_processing": (
        "• Highlight ingredient transparency and simple processing"
        " techniques.\n• Reframe processing steps in terms of natural"
        " fermentation, traditional cooking, or wholesome food craft."
    ),
    "naturalness_perceptions": (
        "• Reduce visual package color saturation (Steiner et al., 2026) to"
        " enhance perceived naturalness.\n• Emphasize plant-origin"
        " ingredients and minimal artificial additives."
    ),
    "taste_expectations": (
        "• Utilize sensory-rich descriptions focusing on umami, texture, and"
        " mouthfeel.\n• Provide chef-led culinary preparation tips to counter"
        " negative taste expectations."
    ),
    "sustainability_differentiation": (
        "• Pair environmental claims with explicit quality and taste assurances"
        " to eliminate the 'sustainability liability' (Kunz et al., 2021)."
    ),
    "anchoring_to_meat": (
        "• Frame products as complementary additions to flexitarian diets"
        " rather than demanding immediate total replacement of meat."
    ),
    "trust_and_credibility": (
        "• Leverage independent third-party certifications and transparent"
        " life-cycle assessment (LCA) data."
    ),
}

sorted_constructs = sorted(zip(DISPLAY_LABELS, mean_probs), key=lambda x: x[1], reverse=True)

print("\nTop Identified Psychological Barriers & Targeted Communication Strategy:")
for label, score in sorted_constructs[:3]:
    print(f"\n[{label.upper()}] (Prevalence: {score*100:.1f}%)")
    if label in interventions:
        print(interventions[label])
    else:
        print("• Provide targeted informational framing addressing consumer risk.")


 PSYCHOLOGICALLY TAILORED INTERVENTION RECOMMENDATIONS (STUDY 3)

Top Identified Psychological Barriers & Targeted Communication Strategy:

[UNCERTAINTY_AND_RISK] (Prevalence: 53.1%)
• Provide targeted informational framing addressing consumer risk.

[DIFFERENTIATION_FROM_MEAT] (Prevalence: 41.4%)
• Apply positive differentiation framing (emphasize unique culinary strengths rather than mimicking meat perfectly).
• Avoid inviting direct organoleptic comparison where the product falls short.

[NATURALNESS_PERCEPTIONS] (Prevalence: 30.7%)
• Reduce visual package color saturation (Steiner et al., 2026) to enhance perceived naturalness.
• Emphasize plant-origin ingredients and minimal artificial additives.


## 12. Save Results

In [13]:
# Save results
df.to_csv(OUTPUT_PATH, index=False)
summary_df = pd.DataFrame({
    "construct": DISPLAY_LABELS,
    "mean_probability_pct": mean_probs * 100,
    "std_probability_pct": std_probs * 100,
})
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\n" + "=" * 70)
print(f"[✔] Analysis Complete. Results saved to: {OUTPUT_PATH}")
print("=" * 70)


[✔] Analysis Complete. Results saved to: data\alt_protein_results.csv
